# LAB S3. Static and contextual word embeddings for text representation
### Natural Language and Information Retrieval
### Degree in Data Science · ETSINF

This notebook solves **Lab Session 3** using the corpus `EXIST2026_training_subset.json`.

We build five text representations:
- three **static** embedding-based document representations,
- two **contextual** embedding-based document representations.

For each representation, we identify the **most similar pair of texts** inside each class:
- `SEXIST`
- `NON-SEXIST`

using **cosine similarity**.

# Static embeddings

Let the corpus be

$$
\mathcal{D} = \{d_1, d_2, \dots, d_N\}
$$

and let each text belong to one class

$$
y_i \in \{	ext{SEXIST}, 	ext{NON-SEXIST}\}.
$$

For a static embedding model, every word $w$ is mapped to a fixed vector

$$
\mathbf{e}(w) \in \mathbb{R}^m.
$$

A document representation is obtained by averaging the embeddings of the words that remain after preprocessing:

$$
\mathbf{x}_i = rac{1}{|W_i|} \sum_{w \in W_i} \mathbf{e}(w),
$$

where $W_i$ is the set of valid tokens of document $d_i$ that are found in the embedding model vocabulary.

In [ ]:
#Installing Gensim library
!pip install -U gensim
!pip install -U nltk
!pip install -U fasttext

## Some libraries

In [ ]:
import fasttext.util
import gensim.downloader as api
from gensim.models.keyedvectors import KeyedVectors
import nltk
from nltk.tokenize import word_tokenize, TweetTokenizer
import numpy as np
import pandas as pd
import re
import json
from sklearn.metrics.pairwise import cosine_similarity

nltk.download("punkt_tab")
nltk.download("stopwords")

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


True

## Load dataset and create the final label

The field `labels_task2_1` contains six annotations (`YES` or `NO`) for each meme. We compute the final class by **majority voting**:

$$
Y_i = \sum_{j=1}^{6} \mathbf{1}(a_{ij}=	ext{YES}), \qquad
N_i = \sum_{j=1}^{6} \mathbf{1}(a_{ij}=	ext{NO}).
$$

Then,

$$
	ext{label}(d_i)=
egin{cases}
	ext{SEXIST}, & 	ext{if } Y_i \ge 4 \
	ext{NON-SEXIST}, & 	ext{if } N_i \ge 4
\end{cases}
$$

In [ ]:
import json
import pandas as pd

filename = "EXIST2026_training_subset.json"

with open(filename, "r", encoding="utf-8") as f:
    raw_data = json.load(f)

rows = []
for _, item in raw_data.items():
    votes = item["labels_task2_1"]
    yes_votes = sum(v == "YES" for v in votes)
    no_votes = sum(v == "NO" for v in votes)

    if yes_votes >= 4:
        final_label = "SEXIST"
    elif no_votes >= 4:
        final_label = "NON-SEXIST"
    else:
        final_label = "TIE"  # Should not happen in this subset

    rows.append({
        "id_EXIST": item["id_EXIST"],
        "text": item["text"],
        "labels_task2_1": votes,
        "yes_votes": yes_votes,
        "no_votes": no_votes,
        "label": final_label,
    })

# The activity states that this subset only contains English texts.
df = pd.DataFrame(rows)
df = df[df["label"] != "TIE"].reset_index(drop=True)

print("Dataset size:", len(df))
print(df["label"].value_counts())
df.head()

## Preprocess and tokenize the corpus for static embeddings

For the **static embedding** models, the activity requires the following preprocessing:
- remove URLs,
- remove hashtags,
- remove user references,
- lowercase,
- tokenize using `word_tokenize`,
- remove English stopwords. fileciteturn3file0

If $T_i$ is the original text, we define the cleaned text as

$$
T_i' = 	ext{tokenize}(	ext{lower}(	ext{removeUsers}(	ext{removeHashtags}(	ext{removeURLs}(T_i))))).
$$

In [ ]:
# Info: nltk.tokenize.word_tokenize(text, language='english', preserve_line=False)
import re
import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords

nltk.download("punkt")
nltk.download("stopwords")

stopw = {
    "english": set(stopwords.words("english")),
    "spanish": set(stopwords.words("spanish")),
}

url_re = re.compile(r"https?://\S+|www\.\S+")
hashtag_re = re.compile(r"#\w+")
user_re = re.compile(r"@\w+")


def preprocess(text):
    text = url_re.sub(" ", text)
    text = hashtag_re.sub(" ", text)
    text = user_re.sub(" ", text)
    text = text.lower().strip()
    return text


def tokenize(text_list, lang="english"):
    result = []
    stop_words = stopw[lang]

    for text in text_list:
        clean_text = preprocess(text)
        tokens = word_tokenize(clean_text, language=lang)
        tokens = [tok for tok in tokens if tok.isalpha()]
        tokens = [tok for tok in tokens if tok not in stop_words]
        result.append(tokens)

    return result


tokenized_text = {
    "english": tokenize(df["text"].tolist(), "english"),
}

print(df["text"].head(3))
print("
Sample tokenized texts:")
for tokens in tokenized_text["english"][:5]:
    print(tokens)

## Text representation using static embeddings

We use the following pre-trained English models required by the activity:
- `word2vec-google-news-300`
- `fasttext-wiki-news-subwords-300`
- `glove-wiki-gigaword-300` fileciteturn3file0

For each model, each text is represented by the **average embedding** of its valid tokens.

### Load the models for English: word2vec, fasttext and glove

In [ ]:
# Download / load the required pre-trained English embedding models from Gensim
import gensim.downloader as api

static_models = {
    "word2vec-google-news-300": api.load("word2vec-google-news-300"),
    "fasttext-wiki-news-subwords-300": api.load("fasttext-wiki-news-subwords-300"),
    "glove-wiki-gigaword-300": api.load("glove-wiki-gigaword-300"),
}

for name, model in static_models.items():
    print(name, model.vector_size)

### Compute document representations from static embeddings

If a document token sequence is

$$
W_i = (w_{i1}, w_{i2}, \dots, w_{in_i}),
$$

its document vector is computed as

$$
\mathbf{x}_i^{(s)} = rac{1}{M_i} \sum_{k=1}^{n_i} \mathbf{e}_s(w_{ik}) \cdot \mathbf{1}(w_{ik} \in V_s),
$$

where:
- $s$ denotes the static embedding model,
- $V_s$ is the vocabulary covered by model $s$,
- $M_i$ is the number of valid tokens found in that vocabulary.

If a text has no valid tokens after preprocessing, we return the zero vector.

In [ ]:
import numpy as np


def gensim_sentence_rep(words, keyedvec):
    """
    Compute the word-embedding representation of a list of tokens by averaging
    the vectors of all tokens found in the model vocabulary.
    """
    dim = keyedvec.vector_size
    avg_vec = np.zeros(dim, dtype=float)
    total_w = 0

    for w in words:
        if w in keyedvec:
            avg_vec += keyedvec[w]
            total_w += 1

    if total_w == 0:
        return np.zeros(dim, dtype=float)

    return avg_vec / total_w


static_representations = {}
for model_name, keyedvec in static_models.items():
    X = np.vstack([
        gensim_sentence_rep(tokens, keyedvec)
        for tokens in tokenized_text["english"]
    ])
    static_representations[model_name] = X
    print(model_name, X.shape)

## Compute cosine similarities for static embeddings

Given two document vectors $\mathbf{x}_i$ and $\mathbf{x}_j$, we use cosine similarity:

$$
\cos(\mathbf{x}_i, \mathbf{x}_j) = rac{\mathbf{x}_i^	op \mathbf{x}_j}{\lVert \mathbf{x}_i Vert \lVert \mathbf{x}_j Vert}.
$$

For each representation, we compute the most similar pair **inside each class separately**.

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity


def most_similar_pair(X, sub_df):
    idx = sub_df.index.to_numpy()
    X_sub = X[idx]
    S = cosine_similarity(X_sub)
    np.fill_diagonal(S, -np.inf)
    i, j = np.unravel_index(np.argmax(S), S.shape)

    return {
        "id1": sub_df.iloc[i]["id_EXIST"],
        "text1": sub_df.iloc[i]["text"],
        "id2": sub_df.iloc[j]["id_EXIST"],
        "text2": sub_df.iloc[j]["text"],
        "similarity": float(S[i, j]),
    }


static_results = {}
for rep_name, X in static_representations.items():
    static_results[rep_name] = {
        "SEXIST": most_similar_pair(X, df[df["label"] == "SEXIST"]),
        "NON-SEXIST": most_similar_pair(X, df[df["label"] == "NON-SEXIST"]),
    }

## Show results

In [ ]:
for rep_name, rep_results in static_results.items():
    print("=" * 80)
    print(rep_name)
    for cls, info in rep_results.items():
        print(f"
Class: {cls}")
        print(f"ID1: {info['id1']}")
        print(f"Text1: {info['text1']}")
        print(f"ID2: {info['id2']}")
        print(f"Text2: {info['text2']}")
        print(f"Cosine similarity: {info['similarity']:.4f}")

# Contextual embeddings

Unlike static embeddings, contextual models produce token representations that depend on the full sentence context.

If a transformer processes a tokenized sequence and returns hidden vectors

$$
\mathbf{h}_1, \mathbf{h}_2, \dots, \mathbf{h}_L,
$$

we represent each text by the output vector associated with the special classification token:

$$
\mathbf{x}_i^{(c)} = \mathbf{h}_{	ext{[CLS]}}
$$

for BERT-like architectures. In practice, this corresponds to the first token representation in the final hidden layer.

In [ ]:
!pip install -U transformers
!pip install -U emoji
#!pip install -U ipywidgets

## Some libraries

In [ ]:
import pandas as pd
import torch
from transformers import AutoModel, AutoTokenizer
from transformers import BertTokenizer, BertModel, RobertaTokenizer, RobertaModel
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

## Contextual models required by the activity

We use the two English transformer models required in the PDF:
- `bert-base-uncased`
- `roberta-base` fileciteturn3file0

The activity explicitly states that **no preprocessing is required** for transformer-based models. We therefore feed the raw text directly to the tokenizer and the model. fileciteturn3file0

In [ ]:
modelnames = {
    "english": ["bert-base-uncased", "roberta-base"],
   # "spanish": ["dccuchile/bert-base-spanish-wwm-uncased"]
}

## Which device to use?

In [ ]:
if torch.backends.mps.is_available():  # Mac M? GPU
    device = torch.device("mps")
elif torch.cuda.is_available():  # Nvidia GPU
    device = torch.device("cuda")
else:  # CPU
    device = torch.device("cpu")
print(device)

## Load the tokenizers and the models

In [ ]:
# Load the contextual tokenizers and models with the generic Hugging Face classes
contextual_tokenizers = {}
contextual_models = {}

for model_name in modelnames["english"]:
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModel.from_pretrained(model_name)
    model.eval()
    model.to(device)

    contextual_tokenizers[model_name] = tokenizer
    contextual_models[model_name] = model

    print(f"Loaded: {model_name}")

## Compute document representations with contextual embeddings

Because the full dataset is large, the PDF recommends processing the texts in **batches**. fileciteturn3file0

For each batch, we:
1. tokenize the raw texts,
2. pad and truncate them,
3. feed them to the transformer,
4. extract the contextual vector of the first token,
5. concatenate all batches.

In [ ]:
batch_size = 16
raw_texts = df["text"].tolist()


def contextual_representations(texts, tokenizer, model, batch_size=16, max_length=128):
    all_cls = []

    for start in range(0, len(texts), batch_size):
        batch = texts[start:start + batch_size]
        encoded = tokenizer(
            batch,
            padding=True,
            truncation=True,
            max_length=max_length,
            return_tensors="pt",
        )
        encoded = {k: v.to(device) for k, v in encoded.items()}

        with torch.no_grad():
            outputs = model(**encoded)
            last_hidden_state = outputs.last_hidden_state
            cls_batch = last_hidden_state[:, 0, :]
            all_cls.append(cls_batch.cpu())

    return torch.cat(all_cls, dim=0).numpy()


contextual_reps = {}
for model_name in modelnames["english"]:
    tokenizer = contextual_tokenizers[model_name]
    model = contextual_models[model_name]
    X = contextual_representations(raw_texts, tokenizer, model, batch_size=batch_size, max_length=128)
    contextual_reps[model_name] = X
    print(model_name, X.shape)

## Compute cosine similarities

In [ ]:
contextual_results = {}
for rep_name, X in contextual_reps.items():
    contextual_results[rep_name] = {
        "SEXIST": most_similar_pair(X, df[df["label"] == "SEXIST"]),
        "NON-SEXIST": most_similar_pair(X, df[df["label"] == "NON-SEXIST"]),
    }

## Show results

In [ ]:
for rep_name, rep_results in contextual_results.items():
    print("=" * 80)
    print(rep_name)
    for cls, info in rep_results.items():
        print(f"
Class: {cls}")
        print(f"ID1: {info['id1']}")
        print(f"Text1: {info['text1']}")
        print(f"ID2: {info['id2']}")
        print(f"Text2: {info['text2']}")
        print(f"Cosine similarity: {info['similarity']:.4f}")